# 03 — NEAT & PPO Training

Train a NEAT agent and a PPO agent on BTC/USDT hourly data from Binance.
Save the trained models to `models/` for use by the rolling-test notebook.

In [ ]:
import os
import pickle
import random
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import neat
import numpy as np
import pandas as pd
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv

from hmats.data.binance import fetch_binance_klines
from hmats.data.features import make_features, standardise
from hmats.data.splits import calendar_split
from hmats.data.trading_env import (
    TradingEnv,
    TradingEnvConfig,
    buy_and_hold_metrics,
    evaluate_policy,
    neat_fitness_from_episode,
)

plt.style.use("seaborn-v0_8-darkgrid")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# Paths — resolve relative to repo root
REPO_ROOT = Path.cwd().parents[2]
if not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = Path.cwd()
MODELS_DIR = REPO_ROOT / "models"
CONFIGS_DIR = REPO_ROOT / "configs"
MODELS_DIR.mkdir(exist_ok=True)

## 1. Fetch data & compute features

In [ ]:
SYMBOL = "BTCUSDT"
INTERVAL = "1h"
START = "2021-01-01"
END = "2026-01-01"

raw = fetch_binance_klines(symbol=SYMBOL, interval=INTERVAL, start=START, end=END)
feat_df = make_features(raw)

# Ensure timezone-naive index for clean splitting
if isinstance(feat_df.index, pd.DatetimeIndex) and feat_df.index.tz is not None:
    feat_df.index = feat_df.index.tz_localize(None)

print(f"Data range: {feat_df.index.min()} → {feat_df.index.max()}")
print(f"Rows: {len(feat_df)}")

## 2. Calendar split & standardise

In [ ]:
train_df, val_df, test_df = calendar_split(feat_df, train_end="2023-12-31", val_end="2024-12-31")
print(f"Split sizes — train: {len(train_df)}, val: {len(val_df)}, test: {len(test_df)}")

mu, sd, (X_train, X_val, X_test), P_train, (P_val, P_test) = standardise(train_df, val_df, test_df)

# Save scaler for inference
np.savez(MODELS_DIR / "feature_scaler.npz", mu=mu, sd=sd)
print(f"Feature dim: {X_train.shape[1]}")

## 3. Train PPO

In [ ]:
ENV_CFG = TradingEnvConfig(fee=0.0005, start_cash=1.0, max_episode_steps=None)
PPO_TIMESTEPS = 200_000


def make_train_env():
    return TradingEnv(X_train, P_train, cfg=ENV_CFG, seed=SEED)


vec_env = DummyVecEnv([make_train_env])
ppo = PPO(
    policy="MlpPolicy",
    env=vec_env,
    verbose=1,
    seed=SEED,
    n_steps=2048,
    batch_size=64,
    learning_rate=3e-4,
    gamma=0.99,
)

t0 = time.time()
ppo.learn(total_timesteps=PPO_TIMESTEPS)
ppo_time = time.time() - t0
print(f"PPO training time: {ppo_time:.1f}s")

ppo.save(str(MODELS_DIR / "ppo_model"))
print(f"PPO model saved to {MODELS_DIR / 'ppo_model.zip'}")

## 4. Train NEAT

In [ ]:
NEAT_POP_SIZE = 50
NEAT_GENERATIONS = 20


class FlushReporter(neat.reporting.BaseReporter):
    def start_generation(self, generation):
        print(f"\n=== Generation {generation} ===")
        sys.stdout.flush()

    def post_evaluate(self, config, population, species, best_genome):
        print(f"Best fitness: {best_genome.fitness:.4f}")
        sys.stdout.flush()

    def end_generation(self, config, population, species_set):
        sys.stdout.flush()


def eval_genomes(genomes, config):
    for _, genome in genomes:
        net = neat.nn.FeedForwardNetwork.create(genome, config)
        fitnesses = []
        for _ in range(5):
            env = TradingEnv(X_train, P_train, cfg=ENV_CFG, seed=np.random.randint(1_000_000_000))
            obs, _ = env.reset()
            done = False
            while not done:
                action = int(np.argmax(net.activate(obs)))
                obs, _, terminated, truncated, _ = env.step(action)
                done = terminated or truncated
            fitnesses.append(neat_fitness_from_episode(env))
        genome.fitness = float(np.mean(fitnesses))


# Use the config from configs/
neat_config_path = str(CONFIGS_DIR / "neat_trading.ini")
config = neat.Config(
    neat.DefaultGenome,
    neat.DefaultReproduction,
    neat.DefaultSpeciesSet,
    neat.DefaultStagnation,
    neat_config_path,
)
# Override pop_size from our constant
config.pop_size = NEAT_POP_SIZE

obs_dim = TradingEnv(X_train, P_train, cfg=ENV_CFG, seed=SEED).observation_space.shape[0]
print(f"NEAT starting... obs_dim={obs_dim}, pop={NEAT_POP_SIZE}, gens={NEAT_GENERATIONS}")

p = neat.Population(config)
p.add_reporter(FlushReporter())
stats = neat.StatisticsReporter()
p.add_reporter(stats)

t0 = time.time()
winner = p.run(eval_genomes, NEAT_GENERATIONS)
neat_time = time.time() - t0
print(f"\nNEAT training time: {neat_time:.1f}s")
print(f"Winner fitness: {winner.fitness:.4f}")

with open(MODELS_DIR / "neat_winner.pkl", "wb") as f:
    pickle.dump(winner, f)
print(f"NEAT winner saved to {MODELS_DIR / 'neat_winner.pkl'}")

## 5. Evaluate on validation set

In [ ]:
# PPO eval
def ppo_act(obs):
    action, _ = ppo.predict(obs, deterministic=True)
    return int(action)


val_env_ppo = TradingEnv(X_val, P_val, cfg=ENV_CFG, seed=SEED)
ppo_val = evaluate_policy(val_env_ppo, ppo_act, seed=SEED)

# NEAT eval
winner_net = neat.nn.FeedForwardNetwork.create(winner, config)


def neat_act(obs):
    return int(np.argmax(winner_net.activate(obs)))


val_env_neat = TradingEnv(X_val, P_val, cfg=ENV_CFG, seed=SEED)
neat_val = evaluate_policy(val_env_neat, neat_act, seed=SEED)

# Buy & hold
bh_val = buy_and_hold_metrics(P_val, fee=ENV_CFG.fee)

# Results table
results = pd.DataFrame(
    [
        {
            "method": "PPO",
            **{k: v for k, v in ppo_val.items() if k not in ("equity_curve", "log_returns")},
        },
        {
            "method": "NEAT",
            **{k: v for k, v in neat_val.items() if k not in ("equity_curve", "log_returns")},
        },
        {
            "method": "Buy&Hold",
            **{k: v for k, v in bh_val.items() if k not in ("equity_curve", "log_returns")},
        },
    ]
)
results

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(ppo_val["equity_curve"], label="PPO")
ax.plot(neat_val["equity_curve"], label="NEAT")
ax.plot(bh_val["equity_curve"], label="Buy & Hold")
ax.set_title(f"Validation Equity Curves — {SYMBOL} {INTERVAL}")
ax.set_xlabel("Step")
ax.set_ylabel("Equity")
ax.legend()
plt.tight_layout()
plt.show()